In [3]:
# a lot of ignore warnings were creating noisy outputs, so filter these away.
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [4]:
# Load BESSTIE dataset and view overall stats
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

dataset = load_dataset("surrey-nlp/BESSTIE-CW-26")

print(dataset)

training_dataframe = dataset['train'].to_pandas()
print(training_dataframe['Sentiment'].value_counts())
print(training_dataframe['Sarcasm'].value_counts())
print(training_dataframe['variety'].value_counts())




DatasetDict({
    train: Dataset({
        features: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm'],
        num_rows: 3747
    })
    validation: Dataset({
        features: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm'],
        num_rows: 313
    })
    test: Dataset({
        features: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm'],
        num_rows: 2183
    })
})
Sentiment
0.0    1907
1.0    1840
Name: count, dtype: int64
Sarcasm
0.0    3223
1.0     524
Name: count, dtype: int64
variety
en-IN    1399
en-UK    1203
en-AU    1145
Name: count, dtype: int64


In [5]:
# What is the optimal number to use for max_features?
from sklearn.metrics import f1_score
import numpy as np

test_dataframe = dataset['test'].to_pandas()
seeds = [1, 2, 3]

for max_features in [1000, 1500, 2000, 5000, 10000, 20000]:
    vectorizer = TfidfVectorizer(max_features=max_features)
    X_train = vectorizer.fit_transform(training_dataframe['text'])
    X_test = vectorizer.transform(test_dataframe['text'])

    sentiment_scores = []
    sarcasm_scores = []
    for seed in seeds:
        lr_sentiment = LogisticRegression(max_iter=1000, random_state=seed)
        lr_sentiment.fit(X_train, training_dataframe['Sentiment'])
        sentiment_scores.append(f1_score(test_dataframe['Sentiment'], lr_sentiment.predict(X_test), average='macro'))

        lr_sarcasm = LogisticRegression(max_iter=1000, random_state=seed)
        lr_sarcasm.fit(X_train, training_dataframe['Sarcasm'])
        sarcasm_scores.append(f1_score(test_dataframe['Sarcasm'], lr_sarcasm.predict(X_test), average='macro'))

    print(f"max_features={max_features:>6} | Sentiment F1: {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f} | Sarcasm F1: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")

max_features=  1000 | Sentiment F1: 0.8231 ± 0.0000 | Sarcasm F1: 0.4734 ± 0.0000
max_features=  1500 | Sentiment F1: 0.8263 ± 0.0000 | Sarcasm F1: 0.4712 ± 0.0000
max_features=  2000 | Sentiment F1: 0.8304 ± 0.0000 | Sarcasm F1: 0.4681 ± 0.0000
max_features=  5000 | Sentiment F1: 0.8331 ± 0.0000 | Sarcasm F1: 0.4651 ± 0.0000
max_features= 10000 | Sentiment F1: 0.8317 ± 0.0000 | Sarcasm F1: 0.4619 ± 0.0000
max_features= 20000 | Sentiment F1: 0.8330 ± 0.0000 | Sarcasm F1: 0.4622 ± 0.0000


Accuracy is the wrong metric for our purposes becuase for sarcasm the dataset is highly imbalanced. The 0.86 accuracy is misleading. The precision and recall for sarcasm is 0.00. The macro score is more useful.

1,500 is the smallest vocabulary found with the highest macro-F1. The below explores the effects of applying combinations of bigrams/unigrams and with and without sublinear TF


In [6]:
# sentiment with 10,000 features, unigrams, and sublinear TF

test_dataframe = dataset['test'].to_pandas()

vectorizer = TfidfVectorizer(max_features=10000, sublinear_tf=True)
X_train = vectorizer.fit_transform(training_dataframe['text'])
X_test = vectorizer.transform(test_dataframe['text'])

# Sentiment
lr_sentiment = LogisticRegression(max_iter=1000)
lr_sentiment.fit(X_train, training_dataframe['Sentiment'])
print("Sentiment:")
print(classification_report(test_dataframe['Sentiment'], lr_sentiment.predict(X_test)))

Sentiment:
              precision    recall  f1-score   support

         0.0       0.82      0.87      0.85      1117
         1.0       0.86      0.80      0.83      1066

    accuracy                           0.84      2183
   macro avg       0.84      0.84      0.84      2183
weighted avg       0.84      0.84      0.84      2183



The following experiments with various combinations of max features, ngrams, default and sublinear TF

In [7]:
from sklearn.metrics import f1_score
import numpy as np

configs = [
    {"name": "unigrams, default TF",   "ngram_range": (1,1), "sublinear_tf": False},
    {"name": "unigrams, sublinear TF", "ngram_range": (1,1), "sublinear_tf": True},
    {"name": "bigrams, default TF",    "ngram_range": (1,2), "sublinear_tf": False},
    {"name": "bigrams, sublinear TF",  "ngram_range": (1,2), "sublinear_tf": True},
]

seeds = [1, 2, 3]
results = []

for max_features in [1000, 1500, 2000, 5000, 10000, 20000]:
    for config in configs:
        vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=config["ngram_range"],
            sublinear_tf=config["sublinear_tf"],
        )
        X_train = vectorizer.fit_transform(training_dataframe['text'])
        X_test = vectorizer.transform(test_dataframe['text'])

        sentiment_scores = []
        sarcasm_scores = []
        for seed in seeds:
            lr_sentiment = LogisticRegression(max_iter=1000, random_state=seed).fit(X_train, training_dataframe['Sentiment'])
            lr_sarcasm = LogisticRegression(max_iter=1000, random_state=seed).fit(X_train, training_dataframe['Sarcasm'])
            sentiment_scores.append(f1_score(test_dataframe['Sentiment'], lr_sentiment.predict(X_test), average='macro'))
            sarcasm_scores.append(f1_score(test_dataframe['Sarcasm'], lr_sarcasm.predict(X_test), average='macro'))

        results.append({
            "max_features": max_features,
            "config": config["name"],
            "sentiment mean": np.mean(sentiment_scores),
            "sentiment std": np.std(sentiment_scores),
            "sarcasm mean": np.mean(sarcasm_scores),
            "sarcasm std": np.std(sarcasm_scores),
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

 max_features                 config  sentiment mean  sentiment std  sarcasm mean  sarcasm std
         1000   unigrams, default TF        0.823103   0.000000e+00      0.473428 0.000000e+00
         1000 unigrams, sublinear TF        0.830003   0.000000e+00      0.477134 0.000000e+00
         1000    bigrams, default TF        0.820382   0.000000e+00      0.479734 0.000000e+00
         1000  bigrams, sublinear TF        0.820449   0.000000e+00      0.479924 0.000000e+00
         1500   unigrams, default TF        0.826274   0.000000e+00      0.471222 0.000000e+00
         1500 unigrams, sublinear TF        0.830419   0.000000e+00      0.464828 0.000000e+00
         1500    bigrams, default TF        0.829050   0.000000e+00      0.471222 0.000000e+00
         1500  bigrams, sublinear TF        0.829050   0.000000e+00      0.474288 5.551115e-17
         2000   unigrams, default TF        0.830437   0.000000e+00      0.468117 5.551115e-17
         2000 unigrams, sublinear TF        0.8341

In [38]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
print(sorted(ENGLISH_STOP_WORDS))
print(f"\nTotal: {len(ENGLISH_STOP_WORDS)} words")

['a', 'about', 'above', 'across', 'after', 'afterwards', 'again', 'against', 'all', 'almost', 'alone', 'along', 'already', 'also', 'although', 'always', 'am', 'among', 'amongst', 'amoungst', 'amount', 'an', 'and', 'another', 'any', 'anyhow', 'anyone', 'anything', 'anyway', 'anywhere', 'are', 'around', 'as', 'at', 'back', 'be', 'became', 'because', 'become', 'becomes', 'becoming', 'been', 'before', 'beforehand', 'behind', 'being', 'below', 'beside', 'besides', 'between', 'beyond', 'bill', 'both', 'bottom', 'but', 'by', 'call', 'can', 'cannot', 'cant', 'co', 'con', 'could', 'couldnt', 'cry', 'de', 'describe', 'detail', 'do', 'done', 'down', 'due', 'during', 'each', 'eg', 'eight', 'either', 'eleven', 'else', 'elsewhere', 'empty', 'enough', 'etc', 'even', 'ever', 'every', 'everyone', 'everything', 'everywhere', 'except', 'few', 'fifteen', 'fifty', 'fill', 'find', 'fire', 'first', 'five', 'for', 'former', 'formerly', 'forty', 'found', 'four', 'from', 'front', 'full', 'further', 'get', 'give

The best result is unigrams with sublinear TF at max_features=20,000 (Sentiment mean of 0.83727).

At every vocabulary size, bigrams are slightly worse than unigrams.
This finding is consistent with the commentary by Bekkerman and Allan, 2003 regarding Reuters and 20 Newsgroups datasets. They found at best the use of bigrams produces an improvement which is "statistically insignificant".
Using Bigrams in Text Categorization by Bekkerman and Allan, 2003
https://ciir.cs.umass.edu/pubfiles/ir-408.pdf

The above enumeration of stopwords reveals many words such as 'no', 'nor', 'not' that are essential for understanding sentiment and sarcasm. Therefore we reject stopword removal.  

Next we apply Logistic Regression with balanced class weight.

In [ ]:
# Balanced class weight
lr_sarcasm = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_sarcasm.fit(X_train, training_dataframe['Sarcasm'])
sarcasm_predictions = lr_sarcasm.predict(X_test)
print("Test dataset Logistic Regression Sarcasm metrics with balanced class weight")
print(classification_report(test_dataframe['Sarcasm'], sarcasm_predictions))


Test dataset Logistic Regression Sarcasm metrics with balanced class weight
              precision    recall  f1-score   support

         0.0       0.92      0.81      0.86      1878
         1.0       0.32      0.55      0.40       305

    accuracy                           0.77      2183
   macro avg       0.62      0.68      0.63      2183
weighted avg       0.83      0.77      0.79      2183



The macro F1 score substantially increases from 0.46 to 0.62

The accuracy and weighted averages are slightly lower, but the macro avg is the most important metric.

Next we compare the balanced class with our own class weight ratio sweep.

For details on class weights see "Logistic Regression in Rare Events Data" by King & Zeng, 2001

In [9]:
from sklearn.metrics import precision_score, recall_score

vectorizer = TfidfVectorizer(max_features=10000, sublinear_tf=True)
X_train = vectorizer.fit_transform(training_dataframe['text'])
X_test = vectorizer.transform(test_dataframe['text'])

# examples from class 0 (sentiment) count with weight 1 in all cases. For examples from class 1 (sarcasm) experiment with values 2, 3, 4, 5, 6 and 8.
results = []
for weight in ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}, {0: 1, 1: 5}, {0: 1, 1: 6}, {0: 1, 1: 8}]:
    lr = LogisticRegression(max_iter=1000, class_weight=weight)
    lr.fit(X_train, training_dataframe['Sarcasm'])
    preds = lr.predict(X_test)
    results.append({
        'weight': str(weight),
        'macro_f1': f1_score(test_dataframe['Sarcasm'], preds, average='macro'),
        'sarc_f1': f1_score(test_dataframe['Sarcasm'], preds, pos_label=1),
        'sarc_precision': precision_score(test_dataframe['Sarcasm'], preds, pos_label=1),
        'sarc_recall': recall_score(test_dataframe['Sarcasm'], preds, pos_label=1),
    })

import pandas as pd
pd.DataFrame(results)

,weight,macro_f1,sarc_f1,sarc_precision,sarc_recall
0,balanced,0.635056,0.423913,0.317073,0.639344
1,"{0: 1, 1: 2}",0.523778,0.130890,0.324675,0.081967
2,"{0: 1, 1: 3}",0.593586,0.281188,0.355000,0.232787
3,"{0: 1, 1: 4}",0.612085,0.337008,0.324242,0.350820
4,"{0: 1, 1: 5}",0.642030,0.405405,0.344828,0.491803
5,"{0: 1, 1: 6}",0.639223,0.414129,0.329457,0.557377
6,"{0: 1, 1: 8}",0.625823,0.410367,0.305958,0.622951


The balanced class has the highest recall and f1 scores for sarcasm, confirming it is essentially optimal. The results demonstrate the precision/recall tradeoff.

GloVe analysis

globe.6B.zip is a large file (822MB) and it can take a few minutes to unzip globe.6B.300d.txt

In [10]:
import os, urllib.request, zipfile

def reporthook(block, block_size, total):
    downloaded = block * block_size
    if total > 0:
        pct = min(downloaded / total * 100, 100)
        print(f'\r  {pct:.1f}%  {downloaded/1e6:.1f} / {total/1e6:.1f} MB', end='', flush=True)

glove_zip = 'glove.6B.zip'

if not os.path.exists('glove.6B.300d.txt'):
    if os.path.exists(glove_zip) and not zipfile.is_zipfile(glove_zip):
        print("Removing corrupt zip file...")
        os.remove(glove_zip)

    if not os.path.exists(glove_zip):
        print("Downloading GloVe embeddings (822 MB)...")
        urllib.request.urlretrieve('http://nlp.stanford.edu/data/glove.6B.zip', glove_zip, reporthook)
        print()

    print("Extracting...")
    with zipfile.ZipFile(glove_zip, 'r') as z:
        z.extractall('.')
    print("Done.")
else:
    print("GloVe already extracted, skipping download.")

GloVe already extracted, skipping download.


In [11]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Load GloVe 300d
glove = {}
with open('glove.6B.300d.txt', 'r') as f:
    for line in f:
        parts = line.split()
        glove[parts[0]] = np.array(parts[1:], dtype=np.float32)

def doc_to_glove(text, glove, dim=300):
    words = text.lower().split()
    vecs = [glove[w] for w in words if w in glove]
    if not vecs:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

X_train_glove = np.array([doc_to_glove(t, glove) for t in training_dataframe['text']])
X_test_glove = np.array([doc_to_glove(t, glove) for t in test_dataframe['text']])

seeds = [1, 2]
sentiment_scores, sarcasm_unbal_scores, sarcasm_bal_scores = [], [], []

for seed in seeds:
    lr = LogisticRegression(max_iter=1000, random_state=seed)
    lr.fit(X_train_glove, training_dataframe['Sentiment'])
    sentiment_scores.append(f1_score(test_dataframe['Sentiment'], lr.predict(X_test_glove), average='macro'))

    lr = LogisticRegression(max_iter=1000, random_state=seed)
    lr.fit(X_train_glove, training_dataframe['Sarcasm'])
    sarcasm_unbal_scores.append(f1_score(test_dataframe['Sarcasm'], lr.predict(X_test_glove), average='macro'))

    lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=seed)
    lr.fit(X_train_glove, training_dataframe['Sarcasm'])
    sarcasm_bal_scores.append(f1_score(test_dataframe['Sarcasm'], lr.predict(X_test_glove), average='macro'))

print(f"GloVe 300d + LR  Sentiment:       {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")
print(f"GloVe 300d + LR  Sarcasm (unbal): {np.mean(sarcasm_unbal_scores):.4f} ± {np.std(sarcasm_unbal_scores):.4f}")
print(f"GloVe 300d + LR  Sarcasm (bal):   {np.mean(sarcasm_bal_scores):.4f} ± {np.std(sarcasm_bal_scores):.4f}")

GloVe 300d + LR  Sentiment:       0.8107 ± 0.0000
GloVe 300d + LR  Sarcasm (unbal): 0.5298 ± 0.0000
GloVe 300d + LR  Sarcasm (bal):   0.6096 ± 0.0000


GloVe comes in 50, 100, 200, and 300 dimensional versions. For completeness let's sweep all sizes.

In [12]:
import pandas as pd
from sklearn.metrics import f1_score

seeds = [1, 2]
results = []

for dim in [50, 100, 200, 300]:
    glove = {}
    with open(f'glove.6B.{dim}d.txt', 'r') as f:
        for line in f:
            parts = line.split()
            glove[parts[0]] = np.array(parts[1:], dtype=np.float32)

    X_train_g = np.array([doc_to_glove(t, glove, dim=dim) for t in training_dataframe['text']])
    X_test_g = np.array([doc_to_glove(t, glove, dim=dim) for t in test_dataframe['text']])

    sentiment_scores, sarcasm_unbal_scores, sarcasm_bal_scores = [], [], []
    for seed in seeds:
        lr = LogisticRegression(max_iter=1000, random_state=seed)
        lr.fit(X_train_g, training_dataframe['Sentiment'])
        sentiment_scores.append(f1_score(test_dataframe['Sentiment'], lr.predict(X_test_g), average='macro'))

        lr = LogisticRegression(max_iter=1000, random_state=seed)
        lr.fit(X_train_g, training_dataframe['Sarcasm'])
        sarcasm_unbal_scores.append(f1_score(test_dataframe['Sarcasm'], lr.predict(X_test_g), average='macro'))

        lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=seed)
        lr.fit(X_train_g, training_dataframe['Sarcasm'])
        sarcasm_bal_scores.append(f1_score(test_dataframe['Sarcasm'], lr.predict(X_test_g), average='macro'))

    results.append({
        'dim': dim,
        'sentiment mean': np.mean(sentiment_scores),
        'sentiment std': np.std(sentiment_scores),
        'sarcasm unbal mean': np.mean(sarcasm_unbal_scores),
        'sarcasm unbal std': np.std(sarcasm_unbal_scores),
        'sarcasm bal mean': np.mean(sarcasm_bal_scores),
        'sarcasm bal std': np.std(sarcasm_bal_scores),
    })

pd.DataFrame(results)

,dim,sentiment mean,sentiment std,sarcasm unbal mean,sarcasm unbal std,sarcasm bal mean,sarcasm bal std
0,50,0.780063,0.0,0.498502,0.0,0.597840,0.0
1,100,0.795555,0.0,0.528485,0.0,0.607469,0.0
2,200,0.802859,0.0,0.532441,0.0,0.612419,0.0
3,300,0.810703,0.0,0.529832,0.0,0.609649,0.0


GloVe + LR Sentiment and GloVe + LR Sarcasm (balanced and unbalanced) results are worse than the equivalent LR tests with TF-IDF.

Next we evaluate SVM:

In [13]:
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
import numpy as np

seeds = [1, 2]
sentiment_scores, sarcasm_scores = [], []

for seed in seeds:
    svm = LinearSVC(max_iter=10000, random_state=seed)
    svm.fit(X_train, training_dataframe['Sentiment'])
    sentiment_scores.append(f1_score(test_dataframe['Sentiment'], svm.predict(X_test), average='macro'))

    svm = LinearSVC(max_iter=10000, class_weight='balanced', random_state=seed)
    svm.fit(X_train, training_dataframe['Sarcasm'])
    sarcasm_scores.append(f1_score(test_dataframe['Sarcasm'], svm.predict(X_test), average='macro'))

print(f"SVM  Sentiment:       {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")
print(f"SVM  Sarcasm (bal):   {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")

SVM  Sentiment:       0.8380 ± 0.0000
SVM  Sarcasm (bal):   0.6012 ± 0.0000


With sublinear TF

In [14]:
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
import numpy as np

vectorizer = TfidfVectorizer(max_features=10000, sublinear_tf=True)
X_train = vectorizer.fit_transform(training_dataframe['text'])
X_test = vectorizer.transform(test_dataframe['text'])

seeds = [1, 2]
sentiment_scores, sarcasm_unbal_scores, sarcasm_bal_scores = [], [], []

for seed in seeds:
    svm = LinearSVC(max_iter=10000, random_state=seed)
    svm.fit(X_train, training_dataframe['Sentiment'])
    sentiment_scores.append(f1_score(test_dataframe['Sentiment'], svm.predict(X_test), average='macro'))

    svm = LinearSVC(max_iter=10000, random_state=seed)
    svm.fit(X_train, training_dataframe['Sarcasm'])
    sarcasm_unbal_scores.append(f1_score(test_dataframe['Sarcasm'], svm.predict(X_test), average='macro'))

    svm = LinearSVC(max_iter=10000, class_weight='balanced', random_state=seed)
    svm.fit(X_train, training_dataframe['Sarcasm'])
    sarcasm_bal_scores.append(f1_score(test_dataframe['Sarcasm'], svm.predict(X_test), average='macro'))

print(f"SVM (sublinear TF)  Sentiment:       {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")
print(f"SVM (sublinear TF)  Sarcasm (unbal): {np.mean(sarcasm_unbal_scores):.4f} ± {np.std(sarcasm_unbal_scores):.4f}")
print(f"SVM (sublinear TF)  Sarcasm (bal):   {np.mean(sarcasm_bal_scores):.4f} ± {np.std(sarcasm_bal_scores):.4f}")

SVM (sublinear TF)  Sentiment:       0.8380 ± 0.0000
SVM (sublinear TF)  Sarcasm (unbal): 0.5647 ± 0.0000
SVM (sublinear TF)  Sarcasm (bal):   0.6012 ± 0.0000


In [15]:
!pip install --upgrade transformers


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


RoBERTa sentiment analysis

In [16]:
# this code is sentiment specific and can be removed later because what follows is a refactored version of this code for sentiment and sarcasm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train = dataset['train'].map(tokenize, batched=True)
tokenized_test = dataset['test'].map(tokenize, batched=True)
tokenized_val = dataset['validation'].map(tokenize, batched=True)

tokenized_train = tokenized_train.map(lambda examples: {'labels': [int(s) for s in examples['Sentiment']]}, batched=True, remove_columns=['Sentiment'])
tokenized_test = tokenized_test.map(lambda examples: {'labels': [int(s) for s in examples['Sentiment']]}, batched=True, remove_columns=['Sentiment'])
tokenized_val = tokenized_val.map(lambda examples: {'labels': [int(s) for s in examples['Sentiment']]}, batched=True, remove_columns=['Sentiment'])

tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_val.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, problem_type="single_label_classification")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    from sklearn.metrics import classification_report
    print(classification_report(labels, preds))
    from sklearn.metrics import f1_score
    return {'macro_f1': f1_score(labels, preds, average='macro')}

# Define TrainingArguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    save_total_limit=1,
)

# Instantiate Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.251469,0.900954
2,No log,0.358010,0.835115
3,0.310793,0.255598,0.926491


              precision    recall  f1-score   support

           0       0.93      0.88      0.90       160
           1       0.88      0.93      0.90       153

    accuracy                           0.90       313
   macro avg       0.90      0.90      0.90       313
weighted avg       0.90      0.90      0.90       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.96      0.71      0.82       160
           1       0.76      0.97      0.85       153

    accuracy                           0.84       313
   macro avg       0.86      0.84      0.84       313
weighted avg       0.86      0.84      0.83       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.93      0.93      0.93       160
           1       0.92      0.93      0.93       153

    accuracy                           0.93       313
   macro avg       0.93      0.93      0.93       313
weighted avg       0.93      0.93      0.93       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=705, training_loss=0.2638026758288661, metrics={'train_runtime': 198.569, 'train_samples_per_second': 56.61, 'train_steps_per_second': 3.55, 'total_flos': 739407843325440.0, 'train_loss': 0.2638026758288661, 'epoch': 3.0})

Test set evaluation

In [17]:
from sklearn.metrics import f1_score

sentiment_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    training_args = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=3,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    predictions = trainer.predict(tokenized_test)
    preds = np.argmax(predictions.predictions, axis=-1)
    score = f1_score(tokenized_test['labels'], preds, average='macro')
    sentiment_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nRoBERTa Sentiment: {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")


Seed: 1


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.186012,0.935913
2,No log,0.177444,0.936086
3,0.301154,0.198976,0.942477


              precision    recall  f1-score   support

           0       0.91      0.97      0.94       160
           1       0.97      0.90      0.93       153

    accuracy                           0.94       313
   macro avg       0.94      0.94      0.94       313
weighted avg       0.94      0.94      0.94       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (11 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.94      0.94      0.94       160
           1       0.94      0.94      0.94       153

    accuracy                           0.94       313
   macro avg       0.94      0.94      0.94       313
weighted avg       0.94      0.94      0.94       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.90      0.90      0.90      1117
           1       0.89      0.89      0.89      1066

    accuracy                           0.90      2183
   macro avg       0.90      0.90      0.90      2183
weighted avg       0.90      0.90      0.90      2183

Seed 2 Macro-F1: 0.8950

RoBERTa Sentiment: 0.8930 ± 0.0020


Common methods for RoBERTa analysis

In [18]:
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

# Common methods
def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    print(classification_report(labels, preds))
    return {'macro_f1': f1_score(labels, preds, average='macro')}

def prepare_dataset(split, task):
    tokenized = split.map(tokenize, batched=True)
    tokenized = tokenized.map(
        lambda examples: {'labels': [int(s) for s in examples[task]]},
        batched=True, remove_columns=[task]
    )
    tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return tokenized

# 3 epochs is a common default for fine-tuning. We could try a higher number of epochs as a further investigation.
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    save_total_limit=1,
)

RoBERTA Sarcasm analysis

In [19]:
from sklearn.metrics import f1_score
import numpy as np

train_sentiment = prepare_dataset(dataset['train'], 'Sentiment')
test_sentiment = prepare_dataset(dataset['test'], 'Sentiment')
val_sentiment = prepare_dataset(dataset['validation'], 'Sentiment')

sentiment_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sentiment_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    training_args = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=3,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer = Trainer(
        model=sentiment_model,
        args=training_args,
        train_dataset=train_sentiment,
        eval_dataset=val_sentiment,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    predictions = trainer.predict(test_sentiment)
    preds = np.argmax(predictions.predictions, axis=-1)
    score = f1_score(test_sentiment['labels'], preds, average='macro')
    sentiment_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nSentiment: {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")


Seed: 1


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.198506,0.923039
2,No log,0.203576,0.929626
3,0.298426,0.240753,0.926506


              precision    recall  f1-score   support

           0       0.90      0.96      0.93       160
           1       0.96      0.88      0.92       153

    accuracy                           0.92       313
   macro avg       0.93      0.92      0.92       313
weighted avg       0.93      0.92      0.92       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (11 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.95      0.94      0.94       160
           1       0.94      0.95      0.94       153

    accuracy                           0.94       313
   macro avg       0.94      0.94      0.94       313
weighted avg       0.94      0.94      0.94       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.90      0.88      0.89      1117
           1       0.88      0.90      0.89      1066

    accuracy                           0.89      2183
   macro avg       0.89      0.89      0.89      2183
weighted avg       0.89      0.89      0.89      2183

Seed 2 Macro-F1: 0.8919

Sentiment: 0.8900 ± 0.0018


In [20]:
from sklearn.metrics import f1_score
import numpy as np

train_sentiment = prepare_dataset(dataset['train'], 'Sentiment')
test_sentiment = prepare_dataset(dataset['test'], 'Sentiment')
val_sentiment = prepare_dataset(dataset['validation'], 'Sentiment')

sentiment_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sentiment_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    training_args = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=3,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer = Trainer(
        model=sentiment_model,
        args=training_args,
        train_dataset=train_sentiment,
        eval_dataset=val_sentiment,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    predictions = trainer.predict(test_sentiment)
    preds = np.argmax(predictions.predictions, axis=-1)
    score = f1_score(test_sentiment['labels'], preds, average='macro')
    sentiment_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nSentiment: {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")


Seed: 1


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.177697,0.929505
2,No log,0.209760,0.929706
3,0.306956,0.243460,0.929695


              precision    recall  f1-score   support

           0       0.91      0.96      0.93       160
           1       0.96      0.90      0.93       153

    accuracy                           0.93       313
   macro avg       0.93      0.93      0.93       313
weighted avg       0.93      0.93      0.93       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (11 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.93      0.94      0.93       160
           1       0.93      0.92      0.93       153

    accuracy                           0.93       313
   macro avg       0.93      0.93      0.93       313
weighted avg       0.93      0.93      0.93       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.89      0.90      0.90      1117
           1       0.90      0.88      0.89      1066

    accuracy                           0.89      2183
   macro avg       0.89      0.89      0.89      2183
weighted avg       0.89      0.89      0.89      2183

Seed 2 Macro-F1: 0.8927

Sentiment: 0.8891 ± 0.0036


The results show a problem with the sarcasm model. Almost everything is being predicted as not sarcasm.

So the model needs adjusting to account for the class imbalance.

In [21]:
from torch import nn
import torch
from sklearn.metrics import f1_score
import numpy as np

train_sarcasm = prepare_dataset(dataset['train'], 'Sarcasm')
test_sarcasm = prepare_dataset(dataset['test'], 'Sarcasm')
val_sarcasm = prepare_dataset(dataset['validation'], 'Sarcasm')

class WeightedTrainer(Trainer):
    # override the Trainer compute_loss
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # model expects only the text inputs, so remove labels
        labels = inputs.pop("labels")
        # pass the text inputs through the model and get the logits outputs
        outputs = model(**inputs)
        logits = outputs.logits

        # divide the number of non-sarcasm examples by sarcasm examples for class 1 (sarcasm). This is about 6.15
        sarcasm_ratio = len(training_dataframe[training_dataframe['Sarcasm']==0]) / len(training_dataframe[training_dataframe['Sarcasm']==1])
        # create a weight tensor with two values, 1.0 for class 0 (not sarcasm) and the sarcasm_ratio for class 1
        weight = torch.tensor([1.0, sarcasm_ratio]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        # calculate the loss by comparing the model predictions against the actual labels
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=3,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = WeightedTrainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )

    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nWeightedTrainer Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")



Seed: 1


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.613110,0.650081
2,No log,0.585120,0.657290
3,0.540322,0.652678,0.677129


              precision    recall  f1-score   support

           0       0.92      0.84      0.88       269
           1       0.35      0.52      0.42        44

    accuracy                           0.80       313
   macro avg       0.63      0.68      0.65       313
weighted avg       0.84      0.80      0.81       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (11 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.93      0.83      0.87       269
           1       0.36      0.61      0.46        44

    accuracy                           0.80       313
   macro avg       0.65      0.72      0.67       313
weighted avg       0.85      0.80      0.82       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.93      0.81      0.87      1878
           1       0.35      0.63      0.45       305

    accuracy                           0.79      2183
   macro avg       0.64      0.72      0.66      2183
weighted avg       0.85      0.79      0.81      2183

Seed 2 Macro-F1: 0.6613

WeightedTrainer Sarcasm: 0.6896 ± 0.0282


In [22]:
from torch import nn
import torch
from sklearn.metrics import f1_score
import numpy as np

train_sarcasm = prepare_dataset(dataset['train'], 'Sarcasm')
test_sarcasm = prepare_dataset(dataset['test'], 'Sarcasm')
val_sarcasm = prepare_dataset(dataset['validation'], 'Sarcasm')

class WeightedTrainer(Trainer):
    # override the Trainer compute_loss
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # divide the number of non-sarcasm examples by sarcasm examples for class 1 (sarcasm). This is about 6.15
        sarcasm_ratio = len(training_dataframe[training_dataframe['Sarcasm']==0]) / len(training_dataframe[training_dataframe['Sarcasm']==1])
        weight = torch.tensor([1.0, sarcasm_ratio]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=3,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = WeightedTrainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )

    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nWeightedTrainer Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")



Seed: 1


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.549901,0.643966
2,No log,0.537380,0.643670
3,0.560610,0.548843,0.688660


              precision    recall  f1-score   support

           0       0.94      0.76      0.84       269
           1       0.33      0.70      0.45        44

    accuracy                           0.75       313
   macro avg       0.63      0.73      0.64       313
weighted avg       0.85      0.75      0.79       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (11 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.94      0.75      0.83       269
           1       0.32      0.70      0.44        44

    accuracy                           0.74       313
   macro avg       0.63      0.73      0.64       313
weighted avg       0.85      0.74      0.78       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.91      0.88      0.89      1878
           1       0.37      0.45      0.40       305

    accuracy                           0.82      2183
   macro avg       0.64      0.66      0.65      2183
weighted avg       0.83      0.82      0.82      2183

Seed 2 Macro-F1: 0.6482

WeightedTrainer Sarcasm: 0.6820 ± 0.0337


In [23]:
from torch import nn
import torch
from sklearn.metrics import f1_score
import numpy as np

class WeightedTrainer(Trainer):
    # override the Trainer compute_loss
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        sarcasm_ratio = 3
        weight = torch.tensor([1.0, sarcasm_ratio]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    # reduce the learning rate from the default 5e-5 to 1e-5
    training_args_sarcasm = TrainingArguments(
        output_dir=f'./results_sarcasm/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=1e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = WeightedTrainer(
        model=sarcasm_model,
        args=training_args_sarcasm,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )

    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nWeightedTrainer (ratio=3, lr=1e-5) Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")



Seed: 1


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.489185,0.651730
2,No log,0.455411,0.662426
3,0.447358,0.665777,0.595646
4,0.447358,0.764755,0.630597
5,0.283494,0.749687,0.649166


              precision    recall  f1-score   support

           0       0.91      0.87      0.89       269
           1       0.37      0.48      0.42        44

    accuracy                           0.81       313
   macro avg       0.64      0.67      0.65       313
weighted avg       0.83      0.81      0.82       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (19 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.91      0.89      0.90       269
           1       0.40      0.45      0.43        44

    accuracy                           0.83       313
   macro avg       0.65      0.67      0.66       313
weighted avg       0.84      0.83      0.83       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.94      0.87      0.91      1878
           1       0.46      0.65      0.54       305

    accuracy                           0.84      2183
   macro avg       0.70      0.76      0.72      2183
weighted avg       0.87      0.84      0.85      2183

Seed 2 Macro-F1: 0.7206

WeightedTrainer (ratio=3, lr=1e-5) Sarcasm: 0.7153 ± 0.0054


RoBERTa performs much better for sentiment, but only slightly better for sarcasm.

To confirm whether the lower LR is needed, run the 3x weight with the default LR:

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = WeightedTrainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )

    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nWeightedTrainer (ratio=3, default lr) Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")



Seed: 1


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.734623,0.568276
2,No log,0.633130,0.622158
3,0.543502,0.659451,0.462199
4,0.543502,0.897651,0.623673
5,0.518752,0.614648,0.612026


              precision    recall  f1-score   support

           0       0.88      0.92      0.90       269
           1       0.29      0.20      0.24        44

    accuracy                           0.82       313
   macro avg       0.58      0.56      0.57       313
weighted avg       0.79      0.82      0.80       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (19 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.89      0.91      0.90       269
           1       0.36      0.30      0.33        44

    accuracy                           0.83       313
   macro avg       0.62      0.60      0.61       313
weighted avg       0.81      0.83      0.82       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.89      0.91      0.90      1878
           1       0.36      0.29      0.32       305

    accuracy                           0.83      2183
   macro avg       0.62      0.60      0.61      2183
weighted avg       0.81      0.83      0.82      2183

Seed 2 Macro-F1: 0.6109

WeightedTrainer (ratio=3, default lr) Sarcasm: 0.6233 ± 0.0124


These results show degraded performance in the form of unstable training. The test set result of 0.62 is the same. But using a lower LR is preferred.

Let's try 2e-5

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    # reduce the learning rate to 2e-5
    training_args_sarcasm = TrainingArguments(
        output_dir=f'./results_sarcasm/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = WeightedTrainer(
        model=sarcasm_model,
        args=training_args_sarcasm,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )

    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nWeightedTrainer (ratio=3, lr=2e-5) Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")



Seed: 1


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.492080,0.630896
2,No log,0.405918,0.670646
3,0.450519,0.963576,0.571438
4,0.450519,1.270953,0.573161
5,0.255335,1.297770,0.617230


              precision    recall  f1-score   support

           0       0.90      0.87      0.89       269
           1       0.35      0.41      0.38        44

    accuracy                           0.81       313
   macro avg       0.62      0.64      0.63       313
weighted avg       0.82      0.81      0.81       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (19 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.90      0.95      0.92       269
           1       0.52      0.32      0.39        44

    accuracy                           0.86       313
   macro avg       0.71      0.63      0.66       313
weighted avg       0.84      0.86      0.85       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.91      0.93      0.92      1878
           1       0.49      0.42      0.45       305

    accuracy                           0.86      2183
   macro avg       0.70      0.68      0.69      2183
weighted avg       0.85      0.86      0.85      2183

Seed 2 Macro-F1: 0.6858

WeightedTrainer (ratio=3, lr=2e-5) Sarcasm: 0.6936 ± 0.0078


In [33]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score
import numpy as np
from datasets import load_dataset

model_name = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

if not 'dataset' in globals():
    dataset = load_dataset("surrey-nlp/BESSTIE-CW-26")

train_sentiment = prepare_dataset(dataset['train'], 'Sentiment')
test_sentiment = prepare_dataset(dataset['test'], 'Sentiment')
val_sentiment = prepare_dataset(dataset['validation'], 'Sentiment')

sentiment_scores = []
for seed in [1, 2, 3]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    # The model was loading in float16 by default. Explicitly set float32
    sentiment_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer = Trainer(
        model=sentiment_model,
        args=training_args_seed,
        train_dataset=train_sentiment,
        eval_dataset=val_sentiment,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    predictions = trainer.predict(test_sentiment)
    preds = np.argmax(predictions.predictions, axis=-1)
    score = f1_score(test_sentiment['labels'], preds, average='macro')
    sentiment_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nDeBERTa Sentiment: {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")



Seed: 1


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.154368,0.948839
2,No log,0.129570,0.958439
3,0.272402,0.187922,0.952069
4,0.272402,0.227982,0.952045
5,0.085273,0.231563,0.942487


              precision    recall  f1-score   support

           0       0.94      0.96      0.95       160
           1       0.95      0.94      0.95       153

    accuracy                           0.95       313
   macro avg       0.95      0.95      0.95       313
weighted avg       0.95      0.95      0.95       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (33 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.94      0.96      0.95       160
           1       0.95      0.93      0.94       153

    accuracy                           0.95       313
   macro avg       0.95      0.95      0.95       313
weighted avg       0.95      0.95      0.95       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.89      0.94      0.92      1117
           1       0.94      0.88      0.91      1066

    accuracy                           0.91      2183
   macro avg       0.91      0.91      0.91      2183
weighted avg       0.91      0.91      0.91      2183

Seed 3 Macro-F1: 0.9118

DeBERTa Sentiment: 0.9090 ± 0.0023


2 seed average DeBERTa sentiment test macro-F1 is 0.9077 ± 0.0016

3 seed average DeBERTa sentiment test macro-F1 is 0.9090 ± 0.0023

In [27]:
from sklearn.metrics import f1_score
import numpy as np

train_sarcasm = prepare_dataset(dataset['train'], 'Sarcasm')
test_sarcasm = prepare_dataset(dataset['test'], 'Sarcasm')
val_sarcasm = prepare_dataset(dataset['validation'], 'Sarcasm')

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        'microsoft/deberta-v3-base',
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = Trainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )
    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nDeBERTa Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")



Seed: 1


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.308651,0.462199
2,No log,0.337834,0.673589
3,0.301161,0.458027,0.541536
4,0.301161,0.616286,0.639039
5,0.121810,0.682822,0.640643


              precision    recall  f1-score   support

           0       0.86      1.00      0.92       269
           1       0.00      0.00      0.00        44

    accuracy                           0.86       313
   macro avg       0.43      0.50      0.46       313
weighted avg       0.74      0.86      0.79       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (19 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.90      0.92      0.91       269
           1       0.45      0.41      0.43        44

    accuracy                           0.85       313
   macro avg       0.68      0.66      0.67       313
weighted avg       0.84      0.85      0.84       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.91      0.94      0.93      1878
           1       0.56      0.46      0.50       305

    accuracy                           0.87      2183
   macro avg       0.74      0.70      0.71      2183
weighted avg       0.86      0.87      0.87      2183

Seed 2 Macro-F1: 0.7150

DeBERTa Sarcasm: 0.7175 ± 0.0025


In [28]:
from sklearn.metrics import f1_score
import numpy as np

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        'microsoft/deberta-v3-base',
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=10,
        seed=seed,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = Trainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )
    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nDeBERTa Sarcasm (10 epochs): {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")



Seed: 1


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.353901,0.462199
2,No log,0.352433,0.699038
3,0.310306,0.409119,0.600959
4,0.310306,0.605071,0.707093
5,0.136182,0.850009,0.627381
6,0.136182,0.867972,0.681617
7,0.036660,0.878450,0.674313
8,0.036660,0.981329,0.657357
9,0.012399,0.991196,0.666094
10,0.012399,0.986705,0.666094


              precision    recall  f1-score   support

           0       0.86      1.00      0.92       269
           1       0.00      0.00      0.00        44

    accuracy                           0.86       313
   macro avg       0.43      0.50      0.46       313
weighted avg       0.74      0.86      0.79       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (39 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.90      0.94      0.92       269
           1       0.50      0.39      0.44        44

    accuracy                           0.86       313
   macro avg       0.70      0.66      0.68       313
weighted avg       0.85      0.86      0.85       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.92      0.91      0.91      1878
           1       0.46      0.48      0.47       305

    accuracy                           0.85      2183
   macro avg       0.69      0.70      0.69      2183
weighted avg       0.85      0.85      0.85      2183

Seed 2 Macro-F1: 0.6915

DeBERTa Sarcasm (10 epochs): 0.6962 ± 0.0047


In [34]:
from sklearn.metrics import f1_score
import numpy as np

sarcasm_scores = []
for seed in [1, 2, 3]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        'microsoft/deberta-v3-base',
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=7,
        seed=seed,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = Trainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )
    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nDeBERTa Sarcasm (7 epochs): {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")



Seed: 1


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.476311,0.462199
2,No log,0.322095,0.659165
3,0.328879,0.448442,0.541040
4,0.328879,0.543158,0.666757
5,0.164949,0.709730,0.613739
6,0.164949,0.818004,0.656044
7,0.062861,0.824558,0.626672


              precision    recall  f1-score   support

           0       0.86      1.00      0.92       269
           1       0.00      0.00      0.00        44

    accuracy                           0.86       313
   macro avg       0.43      0.50      0.46       313
weighted avg       0.74      0.86      0.79       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (45 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.90      0.94      0.92       269
           1       0.48      0.34      0.40        44

    accuracy                           0.86       313
   macro avg       0.69      0.64      0.66       313
weighted avg       0.84      0.86      0.85       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.93      0.91      0.92      1878
           1       0.49      0.57      0.53       305

    accuracy                           0.86      2183
   macro avg       0.71      0.74      0.72      2183
weighted avg       0.87      0.86      0.86      2183

Seed 3 Macro-F1: 0.7232

DeBERTa Sarcasm (7 epochs): 0.7130 ± 0.0136


In [31]:
from torch import nn
import torch
from sklearn.metrics import f1_score
import numpy as np

class DeBERTaWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        sarcasm_ratio = len(training_dataframe[training_dataframe['Sarcasm']==0]) / len(training_dataframe[training_dataframe['Sarcasm']==1])
        weight = torch.tensor([1.0, sarcasm_ratio]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        'microsoft/deberta-v3-base',
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        learning_rate=1e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = DeBERTaWeightedTrainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )
    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nDeBERTa WeightedTrainer (lr=1e-5) Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")



Seed: 1


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.538722,0.684394
2,No log,0.491445,0.708360
3,0.475290,1.024773,0.666601
4,0.475290,1.209985,0.635587
5,0.263917,1.262400,0.659165


              precision    recall  f1-score   support

           0       0.94      0.82      0.88       269
           1       0.38      0.68      0.49        44

    accuracy                           0.80       313
   macro avg       0.66      0.75      0.68       313
weighted avg       0.86      0.80      0.82       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (19 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.91      0.93      0.92       269
           1       0.50      0.41      0.45        44

    accuracy                           0.86       313
   macro avg       0.70      0.67      0.68       313
weighted avg       0.85      0.86      0.85       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.93      0.88      0.91      1878
           1       0.45      0.60      0.51       305

    accuracy                           0.84      2183
   macro avg       0.69      0.74      0.71      2183
weighted avg       0.86      0.84      0.85      2183

Seed 2 Macro-F1: 0.7106

DeBERTa WeightedTrainer (lr=1e-5) Sarcasm: 0.7064 ± 0.0041


In [32]:
from torch import nn
import torch
from sklearn.metrics import f1_score
import numpy as np

class DeBERTaWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        sarcasm_ratio = len(training_dataframe[training_dataframe['Sarcasm']==0]) / len(training_dataframe[training_dataframe['Sarcasm']==1])
        weight = torch.tensor([1.0, sarcasm_ratio]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        'microsoft/deberta-v3-base',
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = DeBERTaWeightedTrainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )
    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nDeBERTa WeightedTrainer (lr=2e-5) Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")



Seed: 1


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.536068,0.653851
2,No log,0.476065,0.697151
3,0.480637,0.803529,0.639039
4,0.480637,1.769325,0.613739
5,0.227753,1.654479,0.678314


              precision    recall  f1-score   support

           0       0.93      0.81      0.87       269
           1       0.35      0.61      0.44        44

    accuracy                           0.78       313
   macro avg       0.64      0.71      0.65       313
weighted avg       0.85      0.78      0.81       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (19 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.89      0.95      0.92       269
           1       0.50      0.30      0.37        44

    accuracy                           0.86       313
   macro avg       0.70      0.62      0.65       313
weighted avg       0.84      0.86      0.84       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.93      0.88      0.91      1878
           1       0.47      0.62      0.53       305

    accuracy                           0.85      2183
   macro avg       0.70      0.75      0.72      2183
weighted avg       0.87      0.85      0.86      2183

Seed 2 Macro-F1: 0.7208

DeBERTa WeightedTrainer (lr=2e-5) Sarcasm: 0.7139 ± 0.0069


Summary of DeBERTa results:

DeBERTa Sarcasm (5 epochs): 0.7175 ± 0.0025

DeBERTa Sarcasm (7 epochs): 0.7130 ± 0.0136 (3 seeds)

DeBERTa Sarcasm (10 epochs): 0.6962 ± 0.0047

DeBERTa WeightedTrainer (lr=1e-5) Sarcasm: 0.7064

DeBERTa WeightedTrainer (lr=2e-5) Sarcasm: 0.7139 ± 0.0069